# Build the `vllm-wheels` dataset (one-time, internet ON)

The submission notebook (`kaggle_submission.ipynb`) runs with **internet off**, which means `pip install vllm` doesn't work there. To pre-stage the wheels, run this notebook **once** with internet on, then convert the output into a Kaggle Dataset.

## Notebook settings

- **Accelerator**: any GPU (we don't actually run vllm here — we only download the right `.whl` files; using GPU just ensures the right CUDA-compatible wheels resolve).
- **Internet**: **ON**.
- **Add Data**: nothing required.

## What this produces

After Run All, `/kaggle/working/vllm-wheels/` will contain `vllm-*.whl` plus every transitive dependency Kaggle's base image is missing.

Then click **Save Version → Save & Run All**, wait for it to finish, open the notebook's **Output** tab, and click **New Dataset** on the `vllm-wheels` folder. Name it `vllm-wheels`.

Attach that dataset to `kaggle_submission.ipynb` and you're fully offline.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

DEST = Path("/kaggle/working/vllm-wheels")
if DEST.exists():
    shutil.rmtree(DEST)
DEST.mkdir(parents=True)

# Pin a known-good vllm version. Bump if the submission notebook needs a newer one.
VLLM_SPEC = "vllm==0.6.3"

# Resolve every wheel vllm pulls in, including transitive deps. We avoid
# --no-deps so any missing dependency in the Kaggle base image gets cached too.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "download",
        VLLM_SPEC,
        "-d", str(DEST),
        "--prefer-binary",
    ]
)

wheels = sorted(DEST.glob("*"))
print(f"\nDownloaded {len(wheels)} files into {DEST}:")
for w in wheels:
    print(f"  {w.name}  ({w.stat().st_size/1e6:.1f} MB)")

## Smoke-test the offline install (optional)

Verifies the cached wheels are sufficient to install vllm with `--no-index`. Comment out if you don't want to actually install it in this notebook.

In [ ]:
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install",
        "--no-index",
        "--find-links", str(DEST),
        "vllm",
    ]
)
import vllm
print("vllm version:", vllm.__version__)